In [26]:
import pandas as pd
import numpy as np
import json

output_results = '../results/output.json'
ground_truth = '../ground_truth.csv'

In [27]:
# read json file
with open(output_results, "r") as f:
    results = json.load(f)
output_results = pd.DataFrame(results)
output_results.head()

,image_path,num_balls,balls
0,development_set/106_png.rf.28ee53acf89d9e7f17b...,9,"[{'number': 7, 'xmin': 0.6166666666666667, 'xm..."
1,development_set/10a_png.rf.bdc9984ba169594ea32...,8,"[{'number': 6, 'xmin': 0.1875, 'xmax': 0.20148..."
2,development_set/110_png.rf.9a38b6057e543f83b58...,3,"[{'number': None, 'xmin': 0.3859375, 'xmax': 0..."
3,development_set/114_png.rf.98b2144c25b9f48816a...,12,"[{'number': 5, 'xmin': 0.6479166666666667, 'xm..."
4,development_set/115_png.rf.ed51d53dd5c384b1f26...,10,"[{'number': 10, 'xmin': 0.7, 'xmax': 0.7182291..."


In [28]:
output_results["image_id"] = output_results["image_path"].apply(lambda x: x.split("/")[-1])
output_results = output_results.drop(columns=['image_path'])
output_results.head()

,num_balls,balls,image_id
0,9,"[{'number': 7, 'xmin': 0.6166666666666667, 'xm...",106_png.rf.28ee53acf89d9e7f17b2fb26185597a0.jpg
1,8,"[{'number': 6, 'xmin': 0.1875, 'xmax': 0.20148...",10a_png.rf.bdc9984ba169594ea32b012098ad10dd.jpg
2,3,"[{'number': None, 'xmin': 0.3859375, 'xmax': 0...",110_png.rf.9a38b6057e543f83b58aa59b9748688b.jpg
3,12,"[{'number': 5, 'xmin': 0.6479166666666667, 'xm...",114_png.rf.98b2144c25b9f48816abd7edb00f365c.jpg
4,10,"[{'number': 10, 'xmin': 0.7, 'xmax': 0.7182291...",115_png.rf.ed51d53dd5c384b1f26a2b6ece52ad69.jpg


In [29]:
ground_truth_data = pd.read_csv(ground_truth)
ground_truth_data.head()

,image_id,num_balls,ball_types
0,3f_png.rf.81c7e132365ef95bb19380ca389025f6.jpg,15,"[0,1,2,3,4,5,6,7,8,9,null,11,12,13,14,15]"
1,4a_png.rf.a6bb5c5706fd8628eb53d34a122cf441.jpg,12,"[0,1,2,3,4,5,6,7,8,null,null,null,12,null,14,15]"
2,5a_png.rf.bae1d48b3d2d96a990799b836ecebcbb.jpg,15,"[0,null,2,3,4,5,6,7,8,9,10,11,12,13,14,15]"
3,6f_png.rf.4dcfc0c556af56e418f2226f40765059.jpg,14,"[0,null,2,3,4,5,6,null,8,9,10,11,12,13,14,15]"
4,7a_png.rf.7cf9f36a358969bd16be866d1570a303.jpg,14,"[0,1,2,3,4,5,6,7,8,9,10,11,null,13,14,15]"


In [34]:
eval_df = output_results.merge(
    ground_truth_data,
    on="image_id",
    how="inner",
    suffixes=("_pred", "_gt")
)
eval_df["ball_types"] = eval_df["ball_types"].apply(json.loads)
eval_df.head()

,num_balls_pred,balls,image_id,num_balls_gt,ball_types
0,9,"[{'number': 7, 'xmin': 0.6166666666666667, 'xm...",106_png.rf.28ee53acf89d9e7f17b2fb26185597a0.jpg,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, None, None, Non..."
1,8,"[{'number': 6, 'xmin': 0.1875, 'xmax': 0.20148...",10a_png.rf.bdc9984ba169594ea32b012098ad10dd.jpg,12,"[0, 1, 2, 3, 4, 5, 6, 7, 8, None, 10, 11, None..."
2,3,"[{'number': None, 'xmin': 0.3859375, 'xmax': 0...",110_png.rf.9a38b6057e543f83b58aa59b9748688b.jpg,2,"[0, None, None, None, None, None, None, None, ..."
3,12,"[{'number': 5, 'xmin': 0.6479166666666667, 'xm...",114_png.rf.98b2144c25b9f48816abd7edb00f365c.jpg,16,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
4,10,"[{'number': 10, 'xmin': 0.7, 'xmax': 0.7182291...",115_png.rf.ed51d53dd5c384b1f26a2b6ece52ad69.jpg,11,"[0, None, 2, None, None, None, 6, None, 8, 9, ..."


In [ ]:
mae = np.mean(np.abs(eval_df["num_balls_pred"] - eval_df["num_balls_gt"]))
accuracy = np.mean(eval_df["num_balls_pred"] == eval_df["num_balls_gt"])

print(f"Ball Count MAE: {mae:.4f}")
print(f"Ball Count Accuracy: {accuracy:.4f}")


def compute_undetected(gt, pred):
    gt_set = set([b for b in gt if b is not None])
    pred_set = set([b["number"] for b in pred if b["number"] is not None])

    if len(gt_set) == 0:
        return 0

    missed = gt_set - pred_set
    return len(missed) / len(gt_set)


def compute_correct(gt, pred):
    gt_set = set([b for b in gt if b is not None])
    pred_set = set([b["number"] for b in pred if b["number"] is not None])

    if len(gt_set) == 0:
        return 0

    correct = gt_set & pred_set
    return len(correct) / len(gt_set)


def compute_incorrect(gt, pred):
    gt_set = set([b for b in gt if b is not None])
    pred_set = set([b["number"] for b in pred if b["number"] is not None])

    if len(pred_set) == 0:
        return 0

    incorrect = pred_set - gt_set
    return len(incorrect) / len(pred_set)


undetected_list = []
correct_list = []
incorrect_list = []

for _, row in eval_df.iterrows():
    gt = row["ball_types"]
    pred = row["balls"]

    undetected_list.append(compute_undetected(gt, pred))
    correct_list.append(compute_correct(gt, pred))
    incorrect_list.append(compute_incorrect(gt, pred))

print(f"Average % of undetected balls: {np.mean(undetected_list):.4f}")
print(f"Average % of correctly classified balls: {np.mean(correct_list):.4f}")
print(f"Average % of incorrectly classified balls: {np.mean(incorrect_list):.4f}")

Ball Count MAE: 2.5200
Ball Count Accuracy: 0.1600
Average % of undetected balls: 0.4729
Average % of correctly classified balls: 0.5271
Average % of incorrectly classified balls: 0.0776
